# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**URL:** [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

_This notebook follows FAIR, referencing all dataset elements by their `@id` fields._

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (not as subscripting, dataset.metadata is an object)
metadata_json = dataset.metadata.to_json()

print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)
print("License: ", dataset.metadata.license)
print("Date Published: ", dataset.metadata.date_published)
print("Keywords: ", dataset.metadata.keywords)
print("Record Sets: ", dataset.metadata.record_set)


## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset should include one or more record sets. We inspect the record sets, then list their associated fields (columns), always referencing by `@id`.

In [ ]:
# List available record sets and their @id
print("Available recordSets and their @id:")
record_sets = dataset.metadata.record_set
for rs in record_sets:
    print("-", rs['@id'], ":", rs['name'])

# Choose the primary record set for exploration
# We'll inspect fields/columns for each record set

for rs in record_sets:
    print(f"\nFields for recordSet @id={rs['@id']} (name={rs['name']}):")
    for field in rs['field']:
        print("  -", field['@id'], "-", field['name'], "- type:", field['dataType'])

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets by @id
dataframes = {}

record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Print columns of the first record set
first_rs_id = record_set_ids[0]
print(f"Columns for record set {first_rs_id}:")
print(dataframes[first_rs_id].columns.tolist())
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We use `@id` to reference each field/column. For demonstration, we'll select the GAD-7 score field and group by demographic attributes such as level_of_education.

In [ ]:
# EDA for the main record set
# Map field names to @id for clarity
main_rs = record_sets[0]
main_rs_id = main_rs['@id']
main_df = dataframes[main_rs_id]

# Identify the GAD-7 score and education fields by @id
fields = {f['name']: f['@id'] for f in main_rs['field']}
gad7_field_id = fields.get('GAD-7 score', fields.get('GAD-7', 'gad7_score'))
education_field_id = fields.get('level_of_education', fields.get('Level of Education', 'level_of_education'))

print(f"Using GAD-7 field (@id={gad7_field_id}) and education field (@id={education_field_id})")

# Filter for survey responses with GAD-7 score > 10 (threshold for moderate anxiety)
if gad7_field_id in main_df.columns:
    threshold = 10
    filtered_df = main_df[main_df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}: {len(filtered_df)} items")
    print(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{gad7_field_id}_normalized"] = (
        (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) /
        filtered_df[gad7_field_id].std()
    )
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by education level, if available
    if education_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(education_field_id)[gad7_field_id].mean().reset_index()
        print(f"Mean GAD-7 score grouped by {education_field_id}:")
        print(grouped_df.head())
else:
    print("GAD-7 field not found in data columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot histograms for GAD-7 scores and compare across education groups, referencing all fields by their `@id`.

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[gad7_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of GAD-7 scores ({gad7_field_id})")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Frequency")
    plt.show()

if education_field_id in main_df.columns and gad7_field_id in main_df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=main_df[education_field_id], y=main_df[gad7_field_id])
    plt.title(f"GAD-7 Scores by Education Level (@id={education_field_id})")
    plt.xlabel("Education Level")
    plt.ylabel("GAD-7 Score")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR² mental health dataset using `mlcroissant` and referenced all record sets and fields by their `@id`.
- Exploratory analysis demonstrated the GAD-7 score distribution and its relation to level of education.
- The dataset provides valuable insights for mental health research, public health policy, and AI applications in Africa.

Further analyses may focus on additional psychological assessment fields (PHQ-9, PSQ) and demographic variables.